In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from ydata_profiling import ProfileReport

from sklearn.model_selection import train_test_split , GridSearchCV 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, OrdinalEncoder 
from sklearn.impute import SimpleImputer

# Building & testing every solo Pipeline the data 

In [2]:
# load processed data
XY_Data= pd.read_csv(r'D:\K_REPO\ITI\LEC 2 task\XY_Data.csv')
XY_Data.head()
# pd.set_option('display.max_columns', None)

,OpenPorchSF,ScreenPorch,MasVnrArea,LotFrontage,BsmtFinSF1,GarageCars,TotalBsmtSF,HalfBath,OverallQual,MiscVal,...,HeatingQC,Electrical,KitchenQual,Functional,FireplaceQu,GarageFinish,GarageQual,PavedDrive,PoolQC,Fence
0,61,0,196.0,65.0,706,2,856,1,7,0,...,Ex,SBrkr,Gd,Typ,NaN,RFn,TA,Y,NaN,NaN
1,0,0,0.0,80.0,978,2,1262,0,6,0,...,Ex,SBrkr,TA,Typ,TA,RFn,TA,Y,NaN,NaN
2,42,0,162.0,68.0,486,2,920,1,7,0,...,Ex,SBrkr,Gd,Typ,TA,RFn,TA,Y,NaN,NaN
3,35,0,0.0,60.0,216,3,756,0,7,0,...,Gd,SBrkr,Gd,Typ,Gd,Unf,TA,Y,NaN,NaN
4,84,0,350.0,84.0,655,3,1145,1,8,0,...,Ex,SBrkr,Gd,Typ,TA,RFn,TA,Y,NaN,NaN


In [3]:
# label features 
Numerical_features= ['Fireplaces', 'OverallQual', 'OverallCond', 'PoolArea', 'MasVnrArea', 'KitchenAbvGr', 'WoodDeckSF', 'SalePrice', 'MiscVal', 'GrLivArea', 'BsmtFinSF1', 'LowQualFinSF', 'ScreenPorch', 'GarageCars', '2ndFlrSF', 'BedroomAbvGr', 'HalfBath', 'BsmtFinSF2', 'EnclosedPorch', 'OpenPorchSF', 'TotalBsmtSF', 'FullBath', 'LotArea', 'BsmtFullBath', 'house_Age', 'BsmtUnfSF', 'BsmtHalfBath', 'LotFrontage', '3SsnPorch'] 


Nominal_features= ['MSSubClass', 'MSZoning', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition'] 


Ordinal_features=  ['LotShape', 'LandSlope', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'PavedDrive', 'PoolQC', 'Fence']

In [4]:
XY_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1452 entries, 0 to 1451
Data columns (total 68 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OpenPorchSF    1452 non-null   int64  
 1   ScreenPorch    1452 non-null   int64  
 2   MasVnrArea     1452 non-null   float64
 3   LotFrontage    1452 non-null   float64
 4   BsmtFinSF1     1452 non-null   int64  
 5   GarageCars     1452 non-null   int64  
 6   TotalBsmtSF    1452 non-null   int64  
 7   HalfBath       1452 non-null   int64  
 8   OverallQual    1452 non-null   int64  
 9   MiscVal        1452 non-null   int64  
 10  LowQualFinSF   1452 non-null   int64  
 11  BsmtHalfBath   1452 non-null   int64  
 12  OverallCond    1452 non-null   int64  
 13  FullBath       1452 non-null   int64  
 14  3SsnPorch      1452 non-null   int64  
 15  GrLivArea      1452 non-null   int64  
 16  Fireplaces     1452 non-null   int64  
 17  BsmtFinSF2     1452 non-null   int64  
 18  KitchenA

## I will make 3 sub-Pipelines for eache tybe of features 

### Numeric Pipeline (Logscale to handle outliers + Standrdization)

In [5]:

Features_to_Log_and_Standrize = [ 'WoodDeckSF', 'LowQualFinSF', 'BsmtUnfSF', 'BsmtFinSF1', '2ndFlrSF', 'PoolArea', 'OpenPorchSF', 'GrLivArea', 'MasVnrArea', '3SsnPorch', 'house_Age', 'LotArea', 'TotalBsmtSF', 'ScreenPorch', 'LotFrontage', 'BsmtFinSF2', 'EnclosedPorch', 'MiscVal']
Features_to_Standrize_only = [ 'OverallCond', 'Fireplaces', 'OverallQual', 'BedroomAbvGr' , 'GarageCars', 'FullBath', 'BsmtHalfBath', 'HalfBath', 'KitchenAbvGr',  'BsmtFullBath']


X = XY_Data.drop('SalePrice', axis=1)
y= XY_Data['SalePrice'].values.reshape(-1,1)
log_y=np.log1p(y)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##### log + Standardization Sub-Pipeline

In [7]:
log_standrize_Pipe = Pipeline([ ('Log_scale', FunctionTransformer(np.log1p)),
                                ('Standrize', StandardScaler())
                                 ])

##### Standardization Only Sub-Pipeline

In [8]:
Standrize_Pipe = Pipeline([  ('Standrize', StandardScaler())  ])

##### testing

In [9]:
transformers=[ ('Log & Standrize', log_standrize_Pipe , Features_to_Log_and_Standrize),
                ('Standrize only', Standrize_Pipe, Features_to_Standrize_only),
                ]
Processor= ColumnTransformer(transformers, remainder='passthrough')
    

In [10]:
Processor

,transformers,"[('Log & Standrize', ...), ('Standrize only', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,func,<ufunc 'log1p'>
,inverse_func,None
,validate,False


###### make sure of transformed Numerical data

In [11]:
explore_Numerical_trans = Processor.fit_transform(X_train, y_train)
print(explore_Numerical_trans.shape)
type(explore_Numerical_trans)

(1161, 67)


numpy.ndarray

In [12]:
Numerical_transfomed_data = pd.DataFrame(explore_Numerical_trans)
Numerical_transfomed_data.head(10)

,0,1,2,3,4,5,6,7,8,9,...,57,58,59,60,61,62,63,64,65,66
0,-0.928208,-0.137808,0.189927,0.648416,1.085432,-0.077865,-1.08143,0.172601,1.173504,-0.134828,...,TA,FuseA,TA,Typ,NaN,Unf,TA,Y,NaN,GdWo
1,-0.928208,-0.137808,-0.4664,0.919702,-0.879748,-0.077865,0.734609,-0.637646,-0.808002,-0.134828,...,TA,SBrkr,TA,Typ,NaN,Unf,TA,Y,NaN,NaN
2,-0.928208,-0.137808,0.436183,-1.394339,1.097311,-0.077865,0.753277,0.250509,1.050272,-0.134828,...,TA,SBrkr,TA,Typ,TA,Unf,TA,Y,NaN,MnPrv
3,0.99925,-0.137808,-0.359433,0.786574,-0.879748,-0.077865,-1.08143,-1.58975,1.09678,-0.134828,...,Ex,SBrkr,Gd,Typ,TA,RFn,TA,Y,NaN,NaN
4,-0.928208,-0.137808,0.071162,0.557622,-0.879748,-0.077865,-1.08143,-1.397547,-0.808002,-0.134828,...,Gd,FuseA,Fa,Maj1,NaN,NaN,NaN,Y,NaN,NaN
5,0.847106,-0.137808,0.464028,-1.394339,1.175826,-0.077865,0.628097,0.242808,-0.808002,-0.134828,...,Ex,SBrkr,Gd,Typ,Gd,Fin,TA,Y,NaN,NaN
6,-0.928208,-0.137808,-0.428372,0.828232,1.316183,12.646948,1.12451,1.984791,1.181836,-0.134828,...,TA,SBrkr,Gd,Typ,TA,RFn,TA,Y,Fa,MnPrv
7,0.827575,-0.137808,-0.652188,0.794589,1.111748,-0.077865,0.939419,0.146841,-0.808002,-0.134828,...,Gd,SBrkr,TA,Typ,TA,Fin,TA,Y,NaN,NaN
8,1.186406,-0.137808,-0.228815,0.412465,1.134777,-0.077865,-1.08143,0.471028,0.919684,-0.134828,...,Gd,SBrkr,TA,Typ,TA,RFn,TA,Y,NaN,NaN
9,-0.928208,-0.137808,-3.107269,0.349681,-0.879748,-0.077865,-1.08143,-1.367771,-0.808002,-0.134828,...,Gd,SBrkr,TA,Typ,NaN,Unf,TA,Y,NaN,NaN


### Nominal Pipeline (Imputation + One Hot Encoder )

In [13]:
Nominal_Pipeline= Pipeline(steps= [ ('imputation with none', SimpleImputer(strategy='constant', fill_value='None')),  
                                ('encoding', OneHotEncoder(handle_unknown='ignore'))    ]   )

In [14]:
XY_Data[Nominal_features].isna().sum()[lambda x : x>0]

MasVnrType      864
GarageType       81
MiscFeature    1398
dtype: int64

##### testing

In [15]:
Nominal_transformer= ColumnTransformer(transformers=[('Nominal Imputer & encoder' , Nominal_Pipeline, Nominal_features)] , remainder='passthrough')

In [16]:
Nominal_transformer

,transformers,"[('Nominal Imputer & encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'constant'
,fill_value,'None'


###### make sure of transformed Nominal data

In [17]:

explore_Nominal_pipeline = Nominal_transformer.fit_transform(X_train , y_train)

In [18]:
explore_Nominal_pipeline.shape

(1161, 209)

In [19]:
Nominal_transformed_data =pd.DataFrame(explore_Nominal_pipeline)
Nominal_transformed_data.head()

,0,1,2,3,4,5,6,7,8,9,...,199,200,201,202,203,204,205,206,207,208
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,TA,FuseA,TA,Typ,NaN,Unf,TA,Y,NaN,GdWo
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,TA,SBrkr,TA,Typ,NaN,Unf,TA,Y,NaN,NaN
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,TA,SBrkr,TA,Typ,TA,Unf,TA,Y,NaN,MnPrv
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Ex,SBrkr,Gd,Typ,TA,RFn,TA,Y,NaN,NaN
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Gd,FuseA,Fa,Maj1,NaN,NaN,NaN,Y,NaN,NaN


### Ordinal data Pipeline (Imputation + OrdinalEncoder)

In [20]:
XY_Data[Ordinal_features].isna().sum()[lambda x : x>0]

BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
FireplaceQu      686
GarageFinish      81
GarageQual        81
PoolQC          1445
Fence           1171
dtype: int64

###### Maps of ordinal encoding

In [21]:
Ordinal_features=  ['LotShape', 'LandSlope', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
     'BsmtFinType2', 'HeatingQC', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'PavedDrive', 'PoolQC', 'Fence']
ordinal_map = {
    'LotShape':     ['Nane','Reg','IR1','IR2','IR3'],
    'LandSlope':    ['Nane','Sev','Mod','Gtl'],
    'ExterQual':    ['Nane','Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['Nane','Po','Fa','TA','Gd','Ex'],
    'BsmtQual':     ['Nane','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['Nane','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['Nane','No','Mn','Av','Gd'],
    'BsmtFinType1': ['Nane','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['Nane','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':    ['Nane','Po','Fa','TA','Gd','Ex'],
    'Electrical':   ['Nane','Mix','FuseP','FuseF','FuseA','SBrkr'],
    'KitchenQual':  ['Nane','Po','Fa','TA','Gd','Ex'],
    'Functional':   ['Nane','Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'FireplaceQu':  ['Nane','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['Nane','Unf','RFn','Fin'],
    'GarageQual':   ['Nane','Po','Fa','TA','Gd','Ex'],
    'PavedDrive':   ['Nane','N','P','Y'],
    'PoolQC':       ['Nane','Fa','TA','Gd','Ex'],
    'Fence':        ['Nane','MnWw','GdWo','MnPrv','GdPrv']
}


In [22]:
Ordinal_Pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Nane')),
    ('ordinal_encoder', OrdinalEncoder(
        categories=[ordinal_map[col] for col in Ordinal_features]
    ))
])

#### testing

In [23]:
Ordinal_transformer= ColumnTransformer(transformers=[
    ('Ordinal data Imputaion & Encoding',   Ordinal_Pipeline,   Ordinal_features)
], remainder='passthrough')

In [27]:
Ordinal_transformer

,transformers,"[('Ordinal data Imputaion & Encoding', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'constant'
,fill_value,'Nane'


In [24]:
Explore_Ordinal_transformed_data = Ordinal_transformer.fit_transform(X_train,y_train)

In [25]:
Explore_Ordinal_transformed_data.shape

(1161, 67)

In [26]:
Explore_Ordinal_transformed_df= pd.DataFrame(Explore_Ordinal_transformed_data)
Explore_Ordinal_transformed_df.head()

,0,1,2,3,4,5,6,7,8,9,...,57,58,59,60,61,62,63,64,65,66
0,1.0,3.0,3.0,3.0,3.0,3.0,1.0,4.0,1.0,3.0,...,MetalSd,MetalSd,BrkFace,CBlock,GasW,Y,Attchd,NaN,WD,Normal
1,1.0,3.0,3.0,3.0,3.0,3.0,1.0,5.0,1.0,3.0,...,MetalSd,MetalSd,NaN,CBlock,GasW,N,Detchd,NaN,ConLD,Normal
2,1.0,3.0,3.0,3.0,4.0,3.0,1.0,1.0,1.0,3.0,...,HdBoard,HdBoard,BrkFace,CBlock,GasA,Y,Attchd,NaN,WD,Normal
3,1.0,3.0,4.0,3.0,4.0,3.0,3.0,6.0,1.0,5.0,...,VinylSd,VinylSd,BrkFace,PConc,GasA,Y,Attchd,NaN,WD,Normal
4,2.0,1.0,2.0,2.0,2.0,1.0,4.0,4.0,1.0,4.0,...,Wd Sdng,Wd Sdng,NaN,BrkTil,GasA,N,NaN,NaN,WD,Normal


# the deployment Pipeline (1 column Transformer)

In [29]:
Deployment_Pipe= ColumnTransformer(transformers=[
    ('log_standrize_Pipe', log_standrize_Pipe, Features_to_Log_and_Standrize ),
    ('Standrize only', Standrize_Pipe, Features_to_Standrize_only),
    ('Nominal Imputer & encoder' , Nominal_Pipeline, Nominal_features),
    ('Ordinal data Imputaion & Encoding',   Ordinal_Pipeline,   Ordinal_features),
    ] , remainder='passthrough')